In [1]:
# Ragas 및 HuggingFace 오픈소스 연동을 위한 필수 라이브러리 설치
!pip install -qU ragas langchain-huggingface transformers accelerate bitsandbytes sentence-transformers nest-asyncio

In [2]:
import nest_asyncio
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from langchain_huggingface import HuggingFacePipeline, HuggingFaceEmbeddings

In [3]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_7650/228170927.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy
/tmp/ipykernel_7650/228170927.py:3: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy


In [10]:
nest_asyncio.apply()

print("⏳ 1. 오픈소스 임베딩 모델 로딩 중...")
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

⏳ 1. 오픈소스 임베딩 모델 로딩 중...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [4]:
print("⏳ 2. 오픈소스 LLM (평가관) 4비트 양자화 로딩 중...")
model_id = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

⏳ 2. 오픈소스 LLM (평가관) 4비트 양자화 로딩 중...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
# ✅ 수정된 부분: BitsAndBytesConfig 객체 생성
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)


In [6]:
# 코랩 T4 GPU 메모리에 맞추기 위해 4bit 로드 (quantization_config 사용)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config
)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [7]:
# LangChain 연동을 위한 파이프라인 구축
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1
)

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [8]:
llm = HuggingFacePipeline(pipeline=pipe)
print("✅ 모델 로딩 완료!")

✅ 모델 로딩 완료!


In [11]:
print("⏳ 3. Ragas 평가 테스트 시작...")

# 테스트용 데이터셋 구성 (사용자 질문, RAG의 답변, 참조한 원본 문맥, 모범 답안)
data = {
    "question": [
        "최근 arXiv에 올라온 RAG 관련 논문들에서 주로 다루는 검색 최적화 기법은 무엇인가요?"
    ],
    "answer": [
        "최근 arXiv 동향을 보면, 벡터 검색의 한계를 보완하기 위해 키워드 검색을 결합하는 하이브리드 검색과 Re-ranking 모델을 도입하는 기법이 주를 이루고 있습니다."
    ],
    "contexts": [
        ["최근 1개월간 arXiv에 등록된 AI 트렌드 리포트에 따르면, 단순 벡터 검색(Dense Retrieval)만으로는 고유 명사 처리에 한계가 있어 BM25 등 스파스 검색을 결합한 하이브리드 검색이 표준화되고 있다. 또한 Cross-Encoder를 활용한 Re-ranking 파이프라인이 필수적으로 요구된다."]
    ],
    "ground_truth": [
        "하이브리드 검색(키워드+벡터)과 Re-ranking 모델의 도입입니다."
    ]
}

dataset = Dataset.from_dict(data)

# Ragas 평가 실행 (오픈소스 LLM과 임베딩 모델 주입)
result = evaluate(
    dataset=dataset,
    metrics=[
        faithfulness,      # 문맥과 답변이 사실적으로 일치하는가?
        answer_relevancy   # 질문에 대해 엉뚱한 소리를 하지 않았는가?
    ],
    llm=llm,
    embeddings=embeddings,
    raise_exceptions=False # 오픈소스 LLM의 파싱 에러로 인한 전체 중단 방지
)

print("\n✅ [최종 평가 결과]")
print(result)

⏳ 3. Ragas 평가 테스트 시작...


Evaluating:   0%|          | 0/2 [00:00<?, ?it/s]

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both


✅ [최종 평가 결과]
{'faithfulness': nan, 'answer_relevancy': nan}
